# JFMO Model Training

Train and evaluate EN-RU language identification model using n-gram approach.

**Prerequisites**: Run `JFMO_dataset_creation.ipynb` first to generate `media_transliterated.csv`

Inspired by [Language Identification for Texts Written in Transliteration](https://ceur-ws.org/Vol-871/paper_2.pdf)

## Imports

In [ ]:
import math
import pickle

from tqdm import tqdm
import pandas as pd
from collections import defaultdict, Counter

## Define N-gram Model

In [ ]:
class NgramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngrams = defaultdict(Counter)

    def train(self, texts):
        for text in tqdm(texts, desc="Training n-grams"):
            text = text.lower() + ' '

            for i in range(len(text) - self.n):
                context = text[i:i+self.n]
                next_char = text[i+self.n]
                self.ngrams[context][next_char] += 1

    def probability(self, text):
        text = text.lower() + ' '
        log_prob = 0.0

        for i in range(len(text) - self.n):
            context = text[i:i+self.n]
            next_char = text[i+self.n]

            count_next = self.ngrams[context][next_char]
            count_total = sum(self.ngrams[context].values())

            if count_total > 0:
                prob = (count_next + 1) / (count_total + 100)
                log_prob += math.log(prob)
            else:
                log_prob += math.log(0.001)

        return log_prob / len(text) if len(text) > 0 else -1000

    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump({'n': self.n, 'ngrams': dict(self.ngrams)}, f)
        print(f"Model saved to: {filepath}")

    def load(self, filepath):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.n = data['n']
            self.ngrams = defaultdict(Counter, data['ngrams'])

## Load Dataset

In [ ]:
print("Loading dataset...")
train_df = pd.read_csv('media_transliterated.csv')

print(f"Total rows: {len(train_df)}")
print(f"Columns: {train_df.columns.tolist()}")
print(f"\nLanguage distribution:")
print(train_df['language'].value_counts())
print(f"\nSample:")
print(train_df.head(10))

## Initialize Models

In [ ]:
model_ru = NgramModel(n=3)
model_en = NgramModel(n=3)

## Train Models

### Russian Model

In [ ]:
ru_texts = train_df[train_df['language'] == 'ru']['title_translit'].dropna().tolist()
print(f"Training RU model with {len(ru_texts)} texts...")
model_ru.train(ru_texts)

### English Model

In [ ]:
en_texts = train_df[train_df['language'] == 'en']['title'].dropna().tolist()
print(f"Training EN model with {len(en_texts)} texts...")
model_en.train(en_texts)

## Save Models

In [ ]:
model_ru.save('jfmo_russian_model.pkl')
model_en.save('jfmo_english_model.pkl')

## Evaluation

### Test Cases

In [ ]:
test_cases = [
    # Russian films - Classics
    ("Brat", True),
    ("Brat 2", True),
    ("Stalker", True),
    ("Solaris", True),
    ("Moskva slezam ne verit", True),
    ("Leviafan", True),
    ("Nochnoy dozor", True),
    ("Dnevnoy dozor", True),
    ("Ironia sudby", True),
    ("Brilliantovaya ruka", True),
    ("Voyna i mir", True),
    ("Ostrov", True),
    ("Vozvrashchenie", True),
    ("Beloe solntse pustyni", True),
    ("Operatsiya Y", True),
    ("Kavkazskaya plennitsa", True),
    ("Ivan Vasilevich", True),
    ("Sluzhebnyy roman", True),
    ("Zerkalo", True),
    ("Andrey Rublev", True),
    ("Ivanovo detstvo", True),
    ("Nostalghiya", True),
    ("Gruz 200", True),
    ("4", True),
    ("Koktebel", True),
    ("Durak", True),

    # English films - Classics
    ("The Matrix", False),
    ("Fight Club", False),
    ("Inception", False),
    ("The Godfather", False),
    ("Pulp Fiction", False),
    ("The Shawshank Redemption", False),
    ("Forrest Gump", False),
    ("Star Wars", False),
    ("Interstellar", False),
    ("The Dark Knight", False),
    ("Titanic", False),
    ("Avatar", False),
    ("Gladiator", False),
    ("Saving Private Ryan", False),
    ("Schindler's List", False),
    ("The Green Mile", False),
    ("The Departed", False),
    ("The Prestige", False),
    ("Memento", False),
    ("The Usual Suspects", False),

    # EDGE CASES
    ("Sputnik", True),
    ("Matrix", False),
    ("Avatar 2", False),
    ("Cheburashka", True),
    ("Rocky", False),
    ("Rambo", False),
    ("Up", False),
    ("Her", False),
    ("It", False),
]

print(f"Total test cases: {len(test_cases)}")

### Run Tests

In [ ]:
correct = 0

print(f"\n{'Text':30} {'Prob_RU':10} {'Prob_EN':10} {'Expected':10} {'Got':10} {'Status':8}")
print("-" * 85)

for text, expected_ru in test_cases:
    prob_ru = model_ru.probability(text)
    prob_en = model_en.probability(text)

    is_russian = prob_ru > prob_en

    status = "✅" if is_russian == expected_ru else "❌"
    if is_russian == expected_ru:
        correct += 1

    exp_str = "RU" if expected_ru else "EN"
    got_str = "RU" if is_russian else "EN"
    print(f"{text:30} {prob_ru:10.2f} {prob_en:10.2f} {exp_str:10} {got_str:10} {status:8}")

print(f"\n{'='*85}")
print(f"Accuracy: {correct}/{len(test_cases)} ({100*correct/len(test_cases):.1f}%)")

## Download Models (for Colab)

In [ ]:
# Uncomment in Google Colab:
# from google.colab import files
# files.download('jfmo_russian_model.pkl')
# files.download('jfmo_english_model.pkl')
# print("Files downloaded")

print("✅ Models trained and saved!")
print("  - jfmo_russian_model.pkl")
print("  - jfmo_english_model.pkl")